In [97]:
import pandas as pd
import requests
from io import StringIO

# Data Retrieval

In [ ]:
import requests
import pandas as pd
from io import StringIO

# 1. NGER datasets (2014-15 to 2023-24)
nger_sources = {
    "2014-15": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0075?select=*",
    "2015-16": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0076?select=*",
    "2016-17": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0077?select=*",
    "2017-18": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0078?select=*",
    "2018-19": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0079?select=*",
    "2019-20": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0080?select=*",
    "2020-21": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0081?select=*",
    "2021-22": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0082?select=*",
    "2022-23": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0083?select=*",
    "2023-24": "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0243?select=*"
}

nger_frames = []

for year, url in nger_sources.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    df_year = pd.DataFrame(data)
    df_year["reporting_year"] = year
    nger_frames.append(df_year)

df_nger = pd.concat(nger_frames, ignore_index=True)


# 2. CER accredited projects
df_accredited = pd.read_csv("power-stations-and-projects-accredited.csv")

# 3. CER committed projects
df_committed = pd.read_csv("power-stations-and-projects-committed.csv")

# 4. CER probable projects
df_probable = pd.read_csv("power-stations-and-projects-probable.csv")


# 5. ABS dataset
abs_url = "https://data.api.abs.gov.au/rest/data/ABS,ABS_REGIONAL_ASGS2021,/..1+2+3+4+5+6+7+8+9.A?startPeriod=2020&dimensionAtObservation=AllDimensions"

headers = {
    "Accept": "application/vnd.sdmx.data+csv;labels=both"
}

response = requests.get(abs_url, headers=headers, timeout=60)
response.raise_for_status()
df_abs = pd.read_csv(StringIO(response.text))


# Clean column names
df_nger.columns = df_nger.columns.str.strip()
df_accredited.columns = df_accredited.columns.str.strip()
df_committed.columns = df_committed.columns.str.strip()
df_probable.columns = df_probable.columns.str.strip()
df_abs.columns = df_abs.columns.str.strip()


# Check shapes
print("df_nger:", df_nger.shape)
print("df_accredited:", df_accredited.shape)
print("df_committed:", df_committed.shape)
print("df_probable:", df_probable.shape)
print("df_abs:", df_abs.shape)
print(df_nger["reporting_year"].value_counts().sort_index())

df_nger: (775, 14)
df_accredited: (36, 8)
df_committed: (36, 5)
df_probable: (59, 4)
df_abs: (14828, 11)


# Data Preprocessing

## clean NGER

## merge CER

In [99]:
# 1. 去掉列名空格
df_accredited.columns = df_accredited.columns.str.strip()
df_committed.columns = df_committed.columns.str.strip()
df_probable.columns = df_probable.columns.str.strip()

# 2. rename
df_accredited_full = df_accredited.rename(columns={
    "Power station name": "project_name",
    "State": "state",
    "Installed capacity (MW)": "capacity_mw",
    "Fuel Source (s)": "fuel_source",
    "Accreditation code": "accreditation_code",
    "Postcode": "postcode",
    "Accreditation start date": "accreditation_start_date",
    "Approval date": "approval_date"
})

df_committed_full = df_committed.rename(columns={
    "Project Name": "project_name",
    "State": "state",
    "MW Capacity": "capacity_mw",
    "Fuel Source": "fuel_source",
    "Committed Date (Month/Year)": "committed_date"
})

df_probable_full = df_probable.rename(columns={
    "Probable Projects": "project_name",
    "State": "state",
    "MW Capacity": "capacity_mw",
    "Fuel Source": "fuel_source"
})

# 3. concat（三个表一起）
df_cer = pd.concat(
    [df_accredited_full, df_committed_full, df_probable_full],
    ignore_index=True
)

# 4. 标记来源
df_cer.loc[df_cer["accreditation_code"].notna(), "project_status"] = "accredited"
df_cer.loc[df_cer["committed_date"].notna(), "project_status"] = "committed"

# probable（两列都为空）
df_cer.loc[
    df_cer["accreditation_code"].isna() & df_cer["committed_date"].isna(),
    "project_status"
] = "probable"

# 5. 查看结果
print(df_cer.head())
print(df_cer.info())

  accreditation_code                                       project_name state  \
0           SRPXQLO8                      DONS FORT - SOLAR W SGU - QLD   QLD   
1           SRPYNSK0                    Mintus Coles - Solar w SGU- NSW   NSW   
2           SRPXQLP1               Ventora - Acacia Ridge - Solar - QLD   QLD   
3           SRPYNSK3         Bayside Aged Care - Solar - Morisset - NSW   NSW   
4           SRPVWAQ0  Matrix Composites and Engineering Pty Ltd - So...    WA   

   postcode capacity_mw fuel_source accreditation_start_date approval_date  \
0    4660.0       0.421       Solar               21/10/2025     6/01/2026   
1    2153.0        0.19       Solar                9/12/2025     6/01/2026   
2    4110.0       0.162       Solar               17/11/2025     6/01/2026   
3    2264.0        0.14       Solar               16/12/2025     8/01/2026   
4    6166.0        0.85       Solar               17/12/2025     8/01/2026   

  committed_date project_status  
0         

In [100]:
# =========================
# NGER
# =========================
print("=== NGER shape ===")
print(df_nger.shape)

print("=== NGER columns ===")
display(df_nger.columns)

print("=== NGER head ===")
display(df_nger.head())


# =========================
# CER（你已经合并好的 df_cer）
# =========================
print("=== CER shape ===")
print(df_cer.shape)

print("=== CER columns ===")
display(df_cer.columns)

print("=== CER head ===")
display(df_cer.head())


# =========================
# ABS
# =========================
print("=== ABS shape ===")
print(df_abs.shape)

print("=== ABS columns ===")
display(df_abs.columns)

print("=== ABS head ===")
display(df_abs.head())


=== NGER shape ===
(775, 14)
=== NGER columns ===


Index(['reportingentity', 'facilityname', 'type', 'state',
       'electricityproductionGJ', 'electricityproductionMWh',
       'totalscope1emissionstCO2e', 'totalscope2emissionstCO2e',
       'totalemissionstCO2e', 'emissionintensitytCO2eMWh', 'gridconnected',
       'grid', 'primaryfuel', 'importantnotes'],
      dtype='object')

=== NGER head ===


,reportingentity,facilityname,type,state,electricityproductionGJ,electricityproductionMWh,totalscope1emissionstCO2e,totalscope2emissionstCO2e,totalemissionstCO2e,emissionintensitytCO2eMWh,gridconnected,grid,primaryfuel,importantnotes
0,ACCIONA ENERGY OCEANIA PTY LTD,Cathedral Rocks Wind Farm,F,SA,481948,133874,57,127.0,184,0.0,On,NEM,Wind,-
1,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,491409,136502,50,218.0,268,0.0,On,NEM,Wind,-
2,ACCIONA ENERGY OCEANIA PTY LTD,Mortlake South Wind Farm,F,VIC,1019352,283153,202,1128.0,1330,0.0,On,NEM,Wind,-
3,ACCIONA ENERGY OCEANIA PTY LTD,Mt Gellibrand Wind Farm,F,VIC,1025451,284847,99,1273.0,1372,0.0,On,NEM,Wind,-
4,ACCIONA ENERGY OCEANIA PTY LTD,Waubra Wind Farm,F,VIC,1954964,543046,186,1114.0,1300,0.0,On,NEM,Wind,-


=== CER shape ===
(131, 10)
=== CER columns ===


...

,accreditation_code,project_name,state,postcode,capacity_mw,fuel_source,accreditation_start_date,approval_date,committed_date,project_status
0,SRPXQLO8,DONS FORT - SOLAR W SGU - QLD,QLD,4660.0,0.421,Solar,21/10/2025,6/01/2026,NaN,accredited
1,SRPYNSK0,Mintus Coles - Solar w SGU- NSW,NSW,2153.0,0.19,Solar,9/12/2025,6/01/2026,NaN,accredited
2,SRPXQLP1,Ventora - Acacia Ridge - Solar - QLD,QLD,4110.0,0.162,Solar,17/11/2025,6/01/2026,NaN,accredited
3,SRPYNSK3,Bayside Aged Care - Solar - Morisset - NSW,NSW,2264.0,0.14,Solar,16/12/2025,8/01/2026,NaN,accredited
4,SRPVWAQ0,Matrix Composites and Engineering Pty Ltd - So...,WA,6166.0,0.85,Solar,17/12/2025,8/01/2026,NaN,accredited


=== ABS shape ===
(14828, 11)
=== ABS columns ===


Index(['DATAFLOW', 'MEASURE: Data Item', 'REGIONTYPE: Region Type',
       'ASGS_2021: Region', 'FREQUENCY: Frequency', 'TIME_PERIOD: Time Period',
       'OBS_VALUE', 'UNIT_MEASURE: Unit of Measure',
       'UNIT_MULT: Unit of Multiplier', 'OBS_STATUS: Observation Status',
       'OBS_COMMENT: Observation Comment'],
      dtype='object')

=== ABS head ===


,DATAFLOW,MEASURE: Data Item,REGIONTYPE: Region Type,ASGS_2021: Region,FREQUENCY: Frequency,TIME_PERIOD: Time Period,OBS_VALUE,UNIT_MEASURE: Unit of Measure,UNIT_MULT: Unit of Multiplier,OBS_STATUS: Observation Status,OBS_COMMENT: Observation Comment
0,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_F_7: Estimated resident population:Females...,STE: States and Territories,6: Tasmania,A: Annual,2021,19599.0,PSNS: Persons,NaN,NaN,NaN
1,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_F_7: Estimated resident population:Females...,STE: States and Territories,6: Tasmania,A: Annual,2022,19247.0,PSNS: Persons,NaN,NaN,NaN
2,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_F_7: Estimated resident population:Females...,STE: States and Territories,6: Tasmania,A: Annual,2023,18912.0,PSNS: Persons,NaN,NaN,NaN
3,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_F_7: Estimated resident population:Females...,STE: States and Territories,6: Tasmania,A: Annual,2024,18478.0,PSNS: Persons,NaN,NaN,NaN
4,ABS:ABS_REGIONAL_ASGS2021(1.5.0),CENSUS_15: Religious affiliation:Buddhism (%),STE: States and Territories,6: Tasmania,A: Annual,2021,1.0,PCT: Percent,NaN,NaN,NaN


In [101]:
import pandas as pd
import re

STOP_TOKENS = {
    # "farm", "wind", "solar", "stage", "creek", "hill",
    # "river", "lake", "park", "energy", "power", "project",
    # "facility", "renewable", "generation", "station",
    # "resource", "recovery", "storage", "pumped", "hydro",
    # "port", "downs", "and", "the", "of"
}

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"\b(nsw|qld|vic|wa|sa|tas|act|nt)\b", " ", x)
    x = re.sub(r"\b(pty ltd|pty|ltd)\b", " ", x)
    x = re.sub(r"[^a-z0-9]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def token_set(x):
    if not x:
        return set()
    return {t for t in x.split() if t not in STOP_TOKENS and not t.isdigit()}

df_nger["name_key"] = df_nger["Facility name"].apply(clean_name)
df_cer["name_key"] = df_cer["project_name"].apply(clean_name)

matches = []

for _, n_row in df_nger.iterrows():
    n_name = n_row["name_key"]
    n_state = n_row["State"]
    n_tokens = token_set(n_name)

    if not n_tokens:
        continue

    for _, c_row in df_cer.iterrows():
        c_name = c_row["name_key"]
        c_state = c_row["state"]
        c_tokens = token_set(c_name)

        if n_state != c_state:
            continue
        if not c_tokens:
            continue

        overlap = n_tokens & c_tokens

        if len(overlap) >= 1:
            matches.append({
                "nger_name": n_row["Facility name"],
                "cer_name": c_row["project_name"],
                "nger_key": n_name,
                "cer_key": c_name,
                "state": n_state,
                "shared_tokens": ", ".join(sorted(overlap)),
                "overlap_count": len(overlap)
            })

df_match = pd.DataFrame(matches)

print("=== 匹配结果示例 ===")
display(df_match.sort_values("overlap_count", ascending=False).head(20))

print("\n=== 匹配统计 ===")
print("NGER 总数:", len(df_nger))
print("CER 总数:", len(df_cer))
print("匹配对数:", len(df_match))

if len(df_match) > 0:
    print("匹配到的 NGER 数:", df_match["nger_name"].nunique())
    print("匹配到的 CER 数:", df_match["cer_name"].nunique())

KeyError: 'Facility name'

In [ ]:
df_abs.to_csv("abs_data.csv", index=False)